# History Coverage Check
**Issue #1 — Can we extend data back to 2021?**

Queries the DeFiLlama Chart API fresh for every pool in `top_pools.csv` and reports the earliest available date.
No API key required.

In [ ]:
import time, pathlib
import requests
import pandas as pd

POOLS_CSV = pathlib.Path("data/top_pools.csv")
CHART_URL = "https://yields.llama.fi/chart/"
DELAY     = 0.5
CUTOFF    = pd.Timestamp("2022-01-01")  # anything before this = reaches 2021

print("Ready.")

## 1. Load Pool List

In [ ]:
pools = pd.read_csv(POOLS_CSV)
print(f"{len(pools)} pools loaded")
pools[["token0", "token1", "fee_tier", "llama_id"]]

## 2. Query API — Find Earliest Date per Pool

In [ ]:
rows = []

for _, pool in pools.iterrows():
    label = f"{pool['token0']}/{pool['token1']} {pool['fee_tier']/1e4:.2f}%"
    print(f"  Fetching {label} ...", end="", flush=True)

    try:
        r = requests.get(CHART_URL + pool["llama_id"], timeout=30)
        r.raise_for_status()
        data = r.json().get("data", [])

        if not data:
            print("  no data")
            earliest = pd.NaT
            latest   = pd.NaT
            n_days   = 0
        else:
            dates    = pd.to_datetime([d["timestamp"][:10] for d in data])
            earliest = dates.min()
            latest   = dates.max()
            n_days   = len(dates)
            print(f"  {n_days} days  |  earliest: {earliest.date()}")

    except Exception as e:
        print(f"  ERROR: {e}")
        earliest = pd.NaT
        latest   = pd.NaT
        n_days   = 0

    rows.append({
        "label":    label,
        "token0":   pool["token0"],
        "token1":   pool["token1"],
        "fee_tier": pool["fee_tier"],
        "address":  pool["address"],
        "llama_id": pool["llama_id"],
        "earliest": earliest,
        "latest":   latest,
        "n_days":   n_days,
    })

    time.sleep(DELAY)

coverage = pd.DataFrame(rows)

## 3. Results — Coverage Summary

In [ ]:
coverage["reaches_2021"] = coverage["earliest"] < CUTOFF

display_cols = ["label", "earliest", "latest", "n_days", "reaches_2021"]
print("=" * 70)
print("ALL POOLS — sorted by earliest date")
print("=" * 70)
print(
    coverage[display_cols]
    .sort_values("earliest")
    .to_string(index=False)
)

n_reach = coverage["reaches_2021"].sum()
print(f"\n{n_reach} of {len(coverage)} pools have data from 2021.")

## 4. Pools That Do NOT Reach 2021

In [ ]:
no_2021 = coverage[~coverage["reaches_2021"]].sort_values("earliest")

if no_2021.empty:
    print("All pools have data from 2021.")
else:
    print(f"{len(no_2021)} pools cannot be tracked back to 2021:\n")
    print(
        no_2021[["label", "earliest", "n_days"]]
        .to_string(index=False)
    )

## 5. Pools That DO Reach 2021

In [ ]:
yes_2021 = coverage[coverage["reaches_2021"]].sort_values("earliest")

if yes_2021.empty:
    print("No pools have data from 2021.")
else:
    print(f"{len(yes_2021)} pools with 2021 data:\n")
    print(
        yes_2021[["label", "earliest", "n_days"]]
        .to_string(index=False)
    )

## 6. Verdict

In [ ]:
global_earliest = coverage["earliest"].min()
print(f"Earliest date available across all pools: {global_earliest.date()}")
print()

if n_reach == len(coverage):
    print("✓ ALL 20 pools can be extended back to 2021. Issue #1 is feasible for the full dataset.")
elif n_reach == 0:
    print("✗ No pools have 2021 data. DeFiLlama coverage does not go back that far.")
else:
    print(f"~ PARTIAL: {n_reach} pools reach 2021, {len(coverage) - n_reach} do not.")
    print("  Extending history is feasible for the subset above, but the rest are limited by pool launch date or DeFiLlama coverage.")